<a href="https://colab.research.google.com/github/PemaDT/CompStat-3.0-Urban_Safety-Framework/blob/main/notebooks/%20M4_Predictive_Modeling_and_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install and import
!pip install scikit-learn pandas pyarrow joblib

In [ ]:
# Mount Drive and Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
import joblib
import os
import gc

CLEAN_DIR = '/content/drive/MyDrive/CompStat3_Project/data/clean'
MODEL_DIR = '/content/drive/MyDrive/CompStat3_Project/data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print("Libraries loaded and Drive mounted!")

Mounted at /content/drive
Libraries loaded and Drive mounted!


In [ ]:
# Load Master Dataset
print(" Loading master dataset...")

master_df = pd.read_parquet(f'{CLEAN_DIR}/master_precinct_daily.parquet')
master_df['date'] = pd.to_datetime(master_df['date'])

print(f"Loaded: {master_df.shape}")
print(f"   Date range: {master_df['date'].min()} → {master_df['date'].max()}")
print(f"   Columns: {master_df.columns.tolist()}")

 Loading master dataset...
Loaded: (149886, 13)
   Date range: 2021-01-01 00:00:00 → 2026-04-15 00:00:00
   Columns: ['date', 'precinct_id', 'abandoned_vehicle', 'blocked_driveway', 'graffiti', 'illegal_parking', 'noise___commercial', 'noise___residential', 'noise___street/sidewalk', 'noise___vehicle', 'sanitation_condition', 'street_light_condition', 'arrest_count']


In [ ]:
# K-Means Clustering (Precinct Disorder Profiles)
# Objective 2: Group precincts into "Disorder Profiles"
# We aggregate complaint totals per precinct, then cluster

print("Building precinct disorder profiles via K-Means...")

# Get complaint columns only (exclude metadata and target)
complaint_cols = [c for c in master_df.columns
                  if c not in ['date', 'precinct_id', 'arrest_count']]

# Aggregate: total complaints per precinct over the full period
precinct_profile = (
    master_df
    .groupby('precinct_id')[complaint_cols]
    .sum()
    .reset_index()
)

# Scale features before clustering
scaler = StandardScaler()
X_cluster = scaler.fit_transform(precinct_profile[complaint_cols])

# K-Means with k=4 (Low, Medium, High, Extreme disorder profiles)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
precinct_profile['disorder_cluster'] = kmeans.fit_predict(X_cluster)

# Map cluster labels back to master dataset
master_df = master_df.merge(
    precinct_profile[['precinct_id', 'disorder_cluster']],
    on='precinct_id',
    how='left'
)

print("K-Means clustering complete!")
print(f"   Cluster distribution:\n{precinct_profile['disorder_cluster'].value_counts().sort_index()}")
print(f"\n   Sample precinct profiles:")
print(precinct_profile[['precinct_id', 'disorder_cluster']].head(10))

Building precinct disorder profiles via K-Means...
K-Means clustering complete!
   Cluster distribution:
disorder_cluster
0    39
1    25
2     6
3     8
Name: count, dtype: int64

   Sample precinct profiles:
  precinct_id  disorder_cluster
0           1                 0
1          10                 0
2         100                 0
3         101                 0
4         102                 1
5         103                 1
6         104                 1
7         105                 0
8         106                 1
9         107                 2


In [ ]:
#Define Target Variable
# Binary classification: 1 = High Arrest Activity, 0 = Low
# Using median as threshold (balanced split)

median_val = master_df['arrest_count'].median()
master_df['target'] = (master_df['arrest_count'] > median_val).astype(int)

print(f"Target variable created")
print(f"   Median arrest count threshold: {median_val}")
print(f"   Class distribution:")
print(master_df['target'].value_counts())
print(f"\n   NIR (Baseline Accuracy): {master_df['target'].value_counts(normalize=True).max():.2%}")

Target variable created
   Median arrest count threshold: 5.0
   Class distribution:
target
0    79087
1    70799
Name: count, dtype: int64

   NIR (Baseline Accuracy): 52.76%


In [ ]:
#Prepare Features & Chronological Split
# Feature matrix — include complaint cols + disorder cluster
feature_cols = complaint_cols + ['disorder_cluster']

X = master_df[feature_cols].fillna(0)
y = master_df['target']

# Chronological split: 2021-2023 Train | 2024-2025 Test
# This matches your proposal methodology exactly
train_mask = master_df['date'] < '2024-01-01'
test_mask  = master_df['date'] >= '2024-01-01'

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f" Chronological train/test split complete")
print(f"   Training set (2021–2023): {X_train.shape[0]:,} rows")
print(f"   Testing set  (2024–2025): {X_test.shape[0]:,} rows")
print(f"   Features used: {len(feature_cols)}")

 Chronological train/test split complete
   Training set (2021–2023): 84,992 rows
   Testing set  (2024–2025): 64,894 rows
   Features used: 11


In [ ]:
# Train Random Forest
print(" Training Random Forest Classifier...")
print("   (This may take 5–10 minutes on large data)\n")

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,          # use all available CPU cores
    max_depth=15,       # prevents overfitting
    min_samples_leaf=5  # smooths out noise
)

rf.fit(X_train, y_train)

print(" Random Forest trained!")

 Training Random Forest Classifier...
   (This may take 5–10 minutes on large data)

 Random Forest trained!


In [ ]:
# Evaluate Model
y_pred = rf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
nir    = y.value_counts(normalize=True).max()

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix (Rows=Actual, Cols=Predicted):\n{cm}")

# Classification Report
print("\nDetailed Performance:")
print(classification_report(y_test, y_pred, target_names=['Low Activity', 'High Activity']))

# Accuracy vs NIR
print(f"Accuracy : {acc:.4f}")
print(f"NIR      : {nir:.4f}")
if acc > nir:
    print("✅ Model beats the No Information Rate — it has real predictive value.")
else:
    print("⚠️ Model does not beat NIR — needs tuning.")

MODEL EVALUATION

Confusion Matrix (Rows=Actual, Cols=Predicted):
[[23208  7395]
 [18947 15344]]

Detailed Performance:
               precision    recall  f1-score   support

 Low Activity       0.55      0.76      0.64     30603
High Activity       0.67      0.45      0.54     34291

     accuracy                           0.59     64894
    macro avg       0.61      0.60      0.59     64894
 weighted avg       0.62      0.59      0.59     64894

Accuracy : 0.5941
NIR      : 0.5276
✅ Model beats the No Information Rate — it has real predictive value.


In [ ]:
#Feature Importance (Leading Indicators)
# This is my key research output (which 311 complaints predict arrests most)
import pandas as pd

feature_importance = pd.DataFrame({
    'feature':   feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("=" * 50)
print("TOP LEADING INDICATOR FEATURES")
print("=" * 50)
print(feature_importance.head(15).to_string(index=False))

# Save feature importance for your final report
feature_importance.to_csv(
    f'{MODEL_DIR}/feature_importance.csv',
    index=False
)
print("\n Feature importance saved!")

TOP LEADING INDICATOR FEATURES
                feature  importance
    noise___residential    0.150668
       blocked_driveway    0.144444
        illegal_parking    0.133898
noise___street/sidewalk    0.119610
       disorder_cluster    0.088656
      abandoned_vehicle    0.080316
     noise___commercial    0.077113
        noise___vehicle    0.073771
 street_light_condition    0.069503
   sanitation_condition    0.033355
               graffiti    0.028667

 Feature importance saved!


In [ ]:
#Saving Everything
# Save the trained model
joblib.dump(rf, f'{MODEL_DIR}/rf_v1_model.pkl')

# Save the clustered master dataset for M5 analysis
master_df.to_parquet(
    f'{CLEAN_DIR}/master_with_clusters.parquet',
    index=False
)

# Save the precinct profiles
precinct_profile.to_csv(
    f'{MODEL_DIR}/precinct_disorder_profiles.csv',
    index=False
)

print(" All M4 outputs saved to Google Drive:")
print(f"   {MODEL_DIR}/rf_v1_model.pkl")
print(f"   {MODEL_DIR}/feature_importance.csv")
print(f"   {MODEL_DIR}/precinct_disorder_profiles.csv")
print(f"   {CLEAN_DIR}/master_with_clusters.parquet")

 All M4 outputs saved to Google Drive:
   /content/drive/MyDrive/CompStat3_Project/data/models/rf_v1_model.pkl
   /content/drive/MyDrive/CompStat3_Project/data/models/feature_importance.csv
   /content/drive/MyDrive/CompStat3_Project/data/models/precinct_disorder_profiles.csv
   /content/drive/MyDrive/CompStat3_Project/data/clean/master_with_clusters.parquet
